# EDA

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:
# PATHS
PROJECT_PATH = Path("../").resolve()
DATA_PATH = PROJECT_PATH / "data"
DATA_FEATURES_PATH = DATA_PATH / "NUSW-NB15_features.csv"
DATA_TESTING_PATH = DATA_PATH / "UNSW_NB15_testing-set.csv"
DATA_TRAINING_SET_PATH = DATA_PATH / "UNSW_NB15_training-set.csv"
DATA_1_PATH = DATA_PATH / "UNSW-NB15_1.csv"
DATA_2_PATH = DATA_PATH / "UNSW-NB15_2.csv"
DATA_3_PATH = DATA_PATH / "UNSW-NB15_3.csv"
DATA_4_PATH = DATA_PATH / "UNSW-NB15_4.csv"
DATA_LIST_EVENTS_PATH = DATA_PATH / "UNSW-NB15_LIST_EVENTS.csv"

## Chargement des dataset

In [3]:
df_data_features = pd.read_csv(DATA_FEATURES_PATH, encoding="cp1252")
df_data_features.head(len(df_data_features))

,No.,Name,Type,Description
0,1,srcip,nominal,Source IP address
1,2,sport,integer,Source port number
2,3,dstip,nominal,Destination IP address
3,4,dsport,integer,Destination port number
4,5,proto,nominal,Transaction protocol
5,6,state,nominal,Indicates to the state and its dependent proto...
6,7,dur,Float,Record total duration
7,8,sbytes,Integer,Source to destination transaction bytes
8,9,dbytes,Integer,Destination to source transaction bytes
9,10,sttl,Integer,Source to destination time to live value


Le dataset "NUSW-NB15_features.csv" contient les noms, types et descriptions des features.

In [4]:
df_data_training_set = pd.read_csv(DATA_TRAINING_SET_PATH, encoding="cp1252")
df_data_training_set.head()

,ï»¿id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.0902,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.0003,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.0051,...,1,3,0,0,0,1,3,0,Normal,0
3,4,0.000006,udp,-,INT,2,0,900,0,166666.6608,...,1,3,0,0,0,2,3,0,Normal,0
4,5,0.000010,udp,-,INT,2,0,2126,0,100000.0025,...,1,3,0,0,0,2,3,0,Normal,0


Le dataset "NUSW-NB15_training-set.csv" est le jeu d'entraînement sur lequel nous entraîneront les algorithmes de clusterisation.

In [5]:
print(
    f"Dimension du dataset training-set : {df_data_training_set.shape[0]} lignes et {df_data_training_set.shape[1]} colonnes"
)

Dimension du dataset training-set : 82332 lignes et 45 colonnes


In [6]:
def compare_columns(df1, df2, name1="df1", name2="df2"):
    cols1 = set([c.lower() for c in df1.columns])
    cols2 = set([c.lower() for c in df2.columns])

    only_in_df1 = cols1 - cols2
    only_in_df2 = cols2 - cols1
    common = cols1 & cols2

    print(f"\nColonnes uniquement dans {name1} ({len(only_in_df1)}):")
    print(sorted(only_in_df1))

    print(f"\nColonnes uniquement dans {name2} ({len(only_in_df2)}):")
    print(sorted(only_in_df2))

    print(f"\nColonnes communes ({len(common)}):")
    print(sorted(common))

    return {"only_in_df1": only_in_df1, "only_in_df2": only_in_df2, "common": common}


_ = compare_columns(
    df_data_features.set_index("Name").T,
    df_data_training_set,
    name1="df_data_features",
    name2="df_data_training_set",
)


Colonnes uniquement dans df_data_features (12):
['ct_src_ ltm', 'dintpkt', 'dmeansz', 'dsport', 'dstip', 'ltime', 'res_bdy_len', 'sintpkt', 'smeansz', 'sport', 'srcip', 'stime']

Colonnes uniquement dans df_data_training_set (8):
['ct_src_ltm', 'dinpkt', 'dmean', 'rate', 'response_body_len', 'sinpkt', 'smean', 'ï»¿id']

Colonnes communes (37):
['ackdat', 'attack_cat', 'ct_dst_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'ct_flw_http_mthd', 'ct_ftp_cmd', 'ct_src_dport_ltm', 'ct_srv_dst', 'ct_srv_src', 'ct_state_ttl', 'dbytes', 'djit', 'dload', 'dloss', 'dpkts', 'dtcpb', 'dttl', 'dur', 'dwin', 'is_ftp_login', 'is_sm_ips_ports', 'label', 'proto', 'sbytes', 'service', 'sjit', 'sload', 'sloss', 'spkts', 'state', 'stcpb', 'sttl', 'swin', 'synack', 'tcprtt', 'trans_depth']


* Colonnes aux nom similaires :

| df_data_features |df_data_training_set |
| :- | :- |
| 'ct_src_ ltm' | 'ct_src_ltm' |
| 'dintpkt' | 'dinpkt' |
| 'dmeansz' | 'dmean' |
| 'res_bdy_len' | 'response_body_len' |
| 'sintpkt' | 'sinpkt' |
| 'smeansz' | 'smean' |

* Features non présentes dans le dataset training_set : 'dsport', 'dstip', 'ltime', 'sport', 'srcip', 'stime'.
* Colonnes non présentes dans les features : 'rate' et 'ï»¿id'.

In [8]:
df_data_training_set = pd.read_csv(DATA_LIST_EVENTS_PATH, encoding="cp1252")
df_data_training_set.head()

,Attack category,Attack subcategory,Number of events
0,normal,NaN,2218761
1,Fuzzers,FTP,558
2,Fuzzers,HTTP,1497
3,Fuzzers,RIP,3550
4,Fuzzers,SMB,5245


Le dataset "NUSW-NB15_LIST_EVENTS.csv" présente la répartition des attaques.